In [ ]:
"""
تجربة مباشرة بالكاميرا - ASL 250 كلمة (موديل gislr_pytorch.py)
حط model.pt (أو model_best.pt) بنفس مجلد السكربت، أو حدد المسار بـ MODEL_PATH.
الكلمات بتنقرأ من نفس ملف الموديل (sign2idx).
تثبيت: pip install torch opencv-python mediapipe numpy
خروج: q   |   تصفير الـ buffer: r
"""
import glob
import os
import warnings
from collections import deque

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# ===================== إعدادات التجربة =====================
MODEL_PATH = "model.pt"     # مثال: r"C:\models\model.pt"  (None = بدوّر بمجلد السكربت)
BUF_LEN = 90            # عدد الفريمات اللي بنخزنها (~3 ثواني)
MIN_HAND_FRAMES = 8     # أقل عدد فريمات لازم تظهر فيها اليد عشان نتنبأ
THRESHOLD = 0.70       # بتدريب label smoothing 0.25 + كاميرا حية الثقة بتكون واطية، ارفعه لو طلعت كلمات غلط
WINDOW = 45             # بنتنبأ على آخر N فريم بس (عشان ما ندخل فريمات الراحة)
SMOOTH = 5              # متوسط آخر N تنبؤات
PREDICT_EVERY = 3       # تنبؤ كل N فريم
NO_HAND_RESET = 20      # إذا اختفت اليدين هالعدد من الفريمات، بنفضي الـ buffer

# اختبار الموديل على ملف من الداتا قبل الكاميرا (مهم للتأكد إن الموديل والـ preprocessing سليمين)
# مثال: TEST_PARQUET = r"C:\data\train_landmark_files\16069\100015657.parquet"; TEST_LABEL = "cloud"
TEST_PARQUET = None
TEST_LABEL = None

# ===================== ثوابت (نفس التدريب بالضبط) =====================
INPUT_SIZE = 64
NUM_CLASSES = 250
N_ROWS, N_DIMS = 543, 3
UNITS, LANDMARK_UNITS, NUM_BLOCKS, NUM_HEADS, MLP_RATIO = 512, 384, 2, 8, 2

LIPS0 = np.array([
    61, 185, 40, 39, 37, 0, 267, 269, 270, 409,
    291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
    95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
])
LEFT_HAND0 = np.arange(468, 489)
RIGHT_HAND0 = np.arange(522, 543)
LEFT_POSE0 = np.array([502, 504, 506, 508, 510])
RIGHT_POSE0 = np.array([503, 505, 507, 509, 511])
LM_LEFT = np.concatenate([LIPS0, LEFT_HAND0, LEFT_POSE0])
LM_RIGHT = np.concatenate([LIPS0, RIGHT_HAND0, RIGHT_POSE0])
N_COLS = len(LM_LEFT)  # 66
LIPS_SL, HAND_SL, POSE_SL = slice(0, 40), slice(40, 61), slice(61, 66)
LH_OFF, POSE_OFF, RH_OFF = 468, 489, 522


# ===================== preprocessing (نفس gislr_pytorch.py) =====================
def _pad_edge(a, left, right):
    return np.concatenate([np.repeat(a[:1], left, axis=0), a, np.repeat(a[-1:], right, axis=0)], axis=0)


def preprocess_sequence(raw):
    """raw: [T, 543, 3] (NaN = مفقود) -> data [64, 66, 3], frame_idxs [64] (-1 = padding)"""
    left_cnt = (~np.isnan(raw[:, LEFT_HAND0])).sum()
    right_cnt = (~np.isnan(raw[:, RIGHT_HAND0])).sum()
    left_dom = left_cnt >= right_cnt
    hand = raw[:, LEFT_HAND0] if left_dom else raw[:, RIGHT_HAND0]
    keep = np.flatnonzero((~np.isnan(hand)).sum(axis=(1, 2)) > 0)

    if len(keep) == 0:
        data = np.zeros((1, N_COLS, N_DIMS), dtype=np.float32)
        fidx = np.zeros(1, dtype=np.float32)
    else:
        data = raw[np.ix_(keep, LM_LEFT if left_dom else LM_RIGHT)].copy()
        fidx = (keep - keep.min()).astype(np.float32)
        if not left_dom:  # نعكس x للإيد والـ pose عشان اليد اليمين تصير زي اليسار
            data[:, HAND_SL.start:, 0] = 1.0 - data[:, HAND_SL.start:, 0]

    n = len(data)
    if n < INPUT_SIZE:
        out = np.zeros((INPUT_SIZE, N_COLS, N_DIMS), dtype=np.float32)
        out[:n] = np.nan_to_num(data, nan=0.0)
        idx = np.full(INPUT_SIZE, -1, dtype=np.float32)
        idx[:n] = fidx
        return out, idx

    if n < INPUT_SIZE ** 2:
        rep = INPUT_SIZE * INPUT_SIZE // n
        data = np.repeat(data, rep, axis=0)
        fidx = np.repeat(fidx, rep, axis=0)
    n = len(data)
    pool = -(-n // INPUT_SIZE)
    pad = pool * INPUT_SIZE - n
    pad_l = pad // 2 + INPUT_SIZE // 2
    pad_r = pad // 2 + INPUT_SIZE // 2 + (pad % 2)
    data = _pad_edge(data, pad_l, pad_r).reshape(INPUT_SIZE, -1, N_COLS, N_DIMS)
    fidx = _pad_edge(fidx, pad_l, pad_r).reshape(INPUT_SIZE, -1)
    data = np.nanmean(data, axis=1)
    fidx = fidx.mean(axis=1)
    return np.nan_to_num(data, nan=0.0).astype(np.float32), fidx.astype(np.float32)


# ===================== الموديل (نفس gislr_pytorch.py) =====================
class LandmarkEmbedding(nn.Module):
    def __init__(self, in_dim, units):
        super().__init__()
        self.empty = nn.Parameter(torch.zeros(units))
        self.fc1 = nn.Linear(in_dim, units, bias=False)
        self.fc2 = nn.Linear(units, units, bias=False)

    def forward(self, x):
        e = self.fc2(F.gelu(self.fc1(x)))
        missing = (x == 0).all(dim=-1, keepdim=True)
        return torch.where(missing, self.empty.to(e.dtype), e)


class Embedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.pos = nn.Embedding(INPUT_SIZE + 1, UNITS)
        self.lips = LandmarkEmbedding(40 * 2, LANDMARK_UNITS)
        self.hand = LandmarkEmbedding(21 * 2, LANDMARK_UNITS)
        self.pose = LandmarkEmbedding(5 * 2, LANDMARK_UNITS)
        self.landmark_weights = nn.Parameter(torch.zeros(3))
        self.fc1 = nn.Linear(LANDMARK_UNITS, UNITS, bias=False)
        self.fc2 = nn.Linear(UNITS, UNITS, bias=False)

    def forward(self, lips, hand, pose, fidx):
        x = torch.stack([self.lips(lips), self.hand(hand), self.pose(pose)], dim=-1)
        x = (x * F.softmax(self.landmark_weights, dim=0).to(x.dtype)).sum(-1)
        x = self.fc2(F.gelu(self.fc1(x)))
        mx = fidx.max(dim=1, keepdim=True).values.clamp(min=1.0)
        pos = (fidx / mx * INPUT_SIZE).long().clamp(0, INPUT_SIZE)
        pos = pos.masked_fill(fidx == -1, INPUT_SIZE)
        return x + self.pos(pos)


class MultiHeadAttention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.h = heads
        self.wq, self.wk, self.wv, self.wo = (nn.Linear(d, d) for _ in range(4))

    def forward(self, x, key_mask):
        B, T, D = x.shape
        split = lambda t: t.view(B, T, self.h, D // self.h).transpose(1, 2)
        q, k, v = split(self.wq(x)), split(self.wk(x)), split(self.wv(x))
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask[:, None, None, :])
        return self.wo(out.transpose(1, 2).reshape(B, T, D))


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.mha = MultiHeadAttention(UNITS, NUM_HEADS)
        self.mlp = nn.Sequential(nn.Linear(UNITS, UNITS * MLP_RATIO), nn.GELU(), nn.Dropout(0.0),
                                 nn.Linear(UNITS * MLP_RATIO, UNITS))

    def forward(self, x, key_mask):
        x = x + self.mha(x, key_mask)
        return x + self.mlp(x)


class SignModel(nn.Module):
    def __init__(self):
        super().__init__()
        for k, n in {"lips": 40, "hand": 21, "pose": 5}.items():
            self.register_buffer(f"{k}_mean", torch.zeros(n, 2))
            self.register_buffer(f"{k}_std", torch.ones(n, 2))
        self.embedding = Embedding()
        self.blocks = nn.ModuleList([Block() for _ in range(NUM_BLOCKS)])
        self.dropout = nn.Dropout(0.0)
        self.head = nn.Linear(UNITS, NUM_CLASSES)

    @staticmethod
    def _norm(v, mean, std):
        return torch.where(v == 0, torch.zeros_like(v), (v - mean) / std).flatten(2)

    def forward(self, frames, fidx):
        mask = fidx != -1
        xy = frames[..., :2]
        lips = self._norm(xy[:, :, LIPS_SL], self.lips_mean, self.lips_std)
        hand = self._norm(xy[:, :, HAND_SL], self.hand_mean, self.hand_std)
        pose = self._norm(xy[:, :, POSE_SL], self.pose_mean, self.pose_std)
        x = self.embedding(lips, hand, pose, fidx)
        for blk in self.blocks:
            x = blk(x, mask)
        m = mask.unsqueeze(-1).to(x.dtype)
        x = (x * m).sum(1) / m.sum(1).clamp(min=1.0)
        return self.head(self.dropout(x))


# ===================== تحميل الموديل =====================
try:
    BASE_DIR = os.path.dirname(os.path.abspath('model.pt'))
except NameError:  # Jupyter
    BASE_DIR = os.getcwd()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if MODEL_PATH is None:
    cands = [os.path.join(BASE_DIR, n) for n in ("model.pt", "model_best.pt")]
    cands += sorted(glob.glob(os.path.join(BASE_DIR, "*.pt")) + glob.glob(os.path.join(BASE_DIR, "*.pth")))
    MODEL_PATH = next((p for p in cands if os.path.exists(p)), None)
if MODEL_PATH is None:
    raise FileNotFoundError("ما لقيت model.pt - حطه بنفس مجلد السكربت أو حدد MODEL_PATH")

ck = torch.load(MODEL_PATH, map_location=device)
sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
model = SignModel().to(device)
try:
    model.load_state_dict(sd)
except RuntimeError:
    print("مفاتيح الموديل:", list(sd.keys())[:15])
    raise
model.eval()

sign2idx = ck.get("sign2idx") if isinstance(ck, dict) else None
if sign2idx:
    WORDS = [None] * NUM_CLASSES
    for w, i in sign2idx.items():
        WORDS[i] = w
else:
    WORDS = [f"class_{i}" for i in range(NUM_CLASSES)]
print(f"تم تحميل {os.path.basename(MODEL_PATH)} | {NUM_CLASSES} كلمة | device={device}")


@torch.no_grad()
def predict(raw):
    data, fidx = preprocess_sequence(raw)
    logits = model(torch.from_numpy(data)[None].to(device), torch.from_numpy(fidx)[None].to(device))
    return F.softmax(logits.float(), dim=1)[0].cpu().numpy()


if TEST_PARQUET:
    import pandas as pd
    _d = pd.read_parquet(TEST_PARQUET, columns=["x", "y", "z"]).to_numpy(np.float32)
    _n = len(_d) // N_ROWS
    _P = predict(_d[:_n * N_ROWS].reshape(_n, N_ROWS, N_DIMS))
    _top = np.argsort(-_P)[:5]
    print("اختبار parquet:", " | ".join(f"{WORDS[i]} {_P[i]*100:.1f}%" for i in _top), "| المتوقع:", TEST_LABEL)

# ===================== MediaPipe =====================
mp_holistic = mp.solutions.holistic
mp_draw = mp.solutions.drawing_utils


def extract(results):
    """نفس ترتيب الداتا: face | left_hand | pose | right_hand، والمفقود = NaN. (543, 3)"""
    a = np.full((543, 3), np.nan, np.float32)
    if results.face_landmarks:
        a[:468] = [[l.x, l.y, l.z] for l in results.face_landmarks.landmark]
    if results.left_hand_landmarks:
        a[LH_OFF:LH_OFF + 21] = [[l.x, l.y, l.z] for l in results.left_hand_landmarks.landmark]
    if results.pose_landmarks:
        a[POSE_OFF:POSE_OFF + 33] = [[l.x, l.y, l.z] for l in results.pose_landmarks.landmark]
    if results.right_hand_landmarks:
        a[RH_OFF:RH_OFF + 21] = [[l.x, l.y, l.z] for l in results.right_hand_landmarks.landmark]
    return a


def hand_visible(a):
    return not (np.isnan(a[LH_OFF:LH_OFF + 21, 0]).all() and np.isnan(a[RH_OFF:RH_OFF + 21, 0]).all())


# ===================== الحلقة الرئيسية =====================
buffer = deque(maxlen=BUF_LEN)
recent = deque(maxlen=SMOOTH)
word, conf_shown, top3, guess = "", 0.0, "", ""
frame_i, no_hand = 0, 0

cap = cv2.VideoCapture(0)
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        # مهم: ما نعمل flip للصورة قبل المعالجة (بيبدّل اليمين باليسار وبخرّب التنبؤ)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = holistic.process(rgb)

        a = extract(results)
        buffer.append(a)
        frame_i += 1

        if hand_visible(a):
            no_hand = 0
        else:
            no_hand += 1
            if no_hand >= NO_HAND_RESET:
                buffer.clear(); recent.clear(); word = ""; top3 = ""; guess = ""

        hand_frames = sum(hand_visible(b) for b in buffer)
        if hand_frames >= MIN_HAND_FRAMES and frame_i % PREDICT_EVERY == 0:
            recent.append(predict(np.stack(list(buffer)[-WINDOW:])))
            P = np.mean(recent, axis=0)
            order = np.argsort(-P)[:3]
            top3 = "  ".join(f"{WORDS[j]} {P[j]*100:.0f}%" for j in order)
            i = int(order[0])
            guess = f"{WORDS[i]} {P[i]*100:.0f}%"
            if frame_i % (PREDICT_EVERY * 10) == 0:
                print(f"hand_frames={hand_frames} | {top3}")
            if P[i] >= THRESHOLD:
                word, conf_shown = WORDS[i], float(P[i])
            else:
                word = ""

        for lm, conn in [(results.pose_landmarks, mp_holistic.POSE_CONNECTIONS),
                         (results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS),
                         (results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)]:
            if lm:
                mp_draw.draw_landmarks(frame, lm, conn)
        cv2.rectangle(frame, (0, 0), (frame.shape[1], 60), (245, 117, 16), -1)
        txt = f"{word}  {conf_shown*100:.0f}%" if word else (f"({guess})" if guess else "...")
        cv2.putText(frame, txt, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 3, cv2.LINE_AA)
        cv2.putText(frame, f"top3: {top3}", (20, frame.shape[0] - 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"hand frames: {hand_frames}/{BUF_LEN}", (20, frame.shape[0] - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.imshow("ASL 250 words", frame)
        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"):
            break
        if k == ord("r"):
            buffer.clear(); recent.clear(); word = ""; top3 = ""; guess = ""

cap.release()
cv2.destroyAllWindows()

: 